# Antimicrobial Peptide Activity Prediction (E. coli) — Explained
A teaching version. Every code cell has a plain-language explanation above it.

## 1. What this project is about

**Antimicrobial peptides (AMPs)** are short chains of amino acids that many living things make to defend themselves against bacteria. They are part of the innate immune system. Unlike most antibiotics, many AMPs kill bacteria by physically punching holes in the bacterial cell membrane, rather than by blocking one specific enzyme.

Why do we care? Bacteria are becoming resistant to our normal antibiotics. When a drug targets one enzyme, a single mutation can make the bacterium immune. AMPs attack the whole membrane, which is much harder for a bacterium to redesign. That makes AMPs a promising source of new antibiotics.

The problem: there are millions of possible peptide sequences, and testing each one in the lab is slow and expensive. So we want a shortcut.

**What this project predicts:** given only the amino-acid sequence of a peptide, can a computer guess whether it will kill *E. coli* (active) or not (inactive)? We turn each sequence into a handful of numbers, show those numbers to a machine learning model, and ask it to sort peptides into active vs inactive.

## 2. Machine-learning words you need first

Read these once. We use them everywhere below.

- **Dataset** — the whole table of peptides we work with. Rows and columns, like an Excel sheet.
- **Sample** — one row. Here, one peptide and everything we know about it.
- **Feature** — one measurable number describing a sample, e.g. the peptide's length or its electric charge. Features are the *inputs* the model learns from.
- **Label** — the answer we want to predict. Here it is 1 (active) or 0 (inactive). Labels are the *output*.
- **Binary classification** — sorting samples into exactly two groups. "Active or inactive" is binary. (If there were three groups it would be multi-class.)
- **Training** — showing the model many peptides *together with* their correct labels so it can learn the pattern. This is like studying with the answer key.
- **Testing** — after training, checking the model on peptides it has never seen, to find out if it really learned or just memorised. This is the exam with no answer key.
- **Overfitting** — when a model memorises the training peptides instead of learning a general rule. It scores well on training data but badly on new data. Like a student who memorises past papers word-for-word and then fails a reworded question.
- **Class imbalance** — when one label is much more common than the other, e.g. if 90% of peptides are inactive. A lazy model could then score 90% just by always saying "inactive", which is useless. We watch for this.

### Step: load the tools

**What this does:** brings in the ready-made Python libraries we need. A library is a toolbox someone else wrote so we don't have to.

**Why:** Python by itself can't read spreadsheets, draw charts, or do machine learning. Each `import` line plugs in one toolbox.

**Line by line:**
- `import re` — text-pattern tool (used to check sequences are spelled with real amino-acid letters).
- `import numpy as np` — fast maths on lists of numbers. `as np` just gives it the short nickname `np` so we type less.
- `import pandas as pd` — handles the data table (reading the CSV, filtering rows). Think of it as Excel inside Python. Nickname `pd`.
- `import matplotlib.pyplot as plt` — draws graphs. Nickname `plt`.
- `import seaborn as sns` — makes nicer-looking graphs on top of matplotlib.
- `import peptides` — the biology toolbox that turns a sequence into numbers (charge, hydrophobicity, etc.).
- the `from sklearn... import ...` lines — **scikit-learn** is the machine-learning toolbox. We pull out only the specific pieces we need: the data-splitter, the three models (`SVC`, `RandomForestClassifier`, `GaussianNB`), the scaling/pipeline helpers, `clone`, and the score functions.
- `sns.set_theme(style="whitegrid")` — a one-time setting so every chart has a clean white background with faint gridlines.

**What you'll see:** nothing. If a cell imports successfully it prints no output. No news is good news. (If a toolbox isn't installed you'd get a red error here naming the missing one.)

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import peptides
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.base import clone
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, ConfusionMatrixDisplay)

sns.set_theme(style="whitegrid")

### Step: settings in one place

**What this does:** defines the few values you might need to change, all at the top, so you never have to hunt through the code.

**Why:** if your CSV names its columns differently, or you want a different activity cutoff, you change it here once instead of in ten places.

**Line by line:**
- `CSV_PATH` — the file name of your data. It must sit in the same folder as this notebook.
- `SEQ_COL`, `MIC_COL`, `UNIT_COL` — the column names in *your* file for the sequence, the MIC value, and the unit. **If the printed columns later don't match these, edit them here.**
- `MIC_THRESHOLD = 32` — the cutoff in µg/mL that separates active from inactive (explained in section 4).
- `AA = "ACDEFGHIKLMNPQRSTVWY"` — the 20 standard amino-acid letters. Used to check sequences and to count composition.
- `RANDOM_STATE = 42` — a fixed "seed" for the random steps. Computers' randomness can be frozen to a number so the experiment gives the **same result every time you run it**. 42 is just a conventional choice; any fixed number works. This is what makes your results reproducible.

**What you'll see:** nothing. These lines only store values for later.

In [ ]:
CSV_PATH = "dbaasp_ecoli.csv"
SEQ_COL  = "SEQUENCE"   # change to match your file
MIC_COL  = "MIC"
UNIT_COL = "UNIT"

MIC_THRESHOLD = 32           # ug/mL; active if MIC <= threshold
AA = "ACDEFGHIKLMNPQRSTVWY"
RANDOM_STATE = 42

### Step: read the data and look at it

**What this does:** opens your CSV file and shows its shape and first few rows.

**Why:** before any analysis you must confirm the file loaded and see how it is laid out — especially the exact column names.

**Line by line:**
- `df = pd.read_csv(CSV_PATH)` — read the CSV into a **DataFrame** (a table). `df` is the usual name for it. `read_csv` is the pandas function that does the reading.
- `print("Columns:", list(df.columns))` — print the list of column headings. This is the line that tells you what to put in `SEQ_COL` / `MIC_COL` / `UNIT_COL`.
- `print("Shape:", df.shape)` — print `(rows, columns)`, so you know how many peptides and how many columns you have.
- `df.head()` — show the first 5 rows as a neat table. Putting it on the last line makes Jupyter display it.

**What you'll see:** the column names, a pair of numbers like `(500, 7)` meaning 500 peptides and 7 columns, and a small preview table. **Check the column names against the Configuration cell now.**

In [ ]:
df = pd.read_csv(CSV_PATH)
print("Columns:", list(df.columns))
print("Shape:", df.shape)
df.head()

### Step: fail loudly if the column names are wrong

**What this does:** checks that the sequence and MIC columns you named actually exist. If not, it stops with a clear message instead of breaking mysteriously later.

**Why:** the single most common reason this kind of notebook "doesn't work" is a mismatched column name. Better to catch it here with a helpful message.

**Line by line:**
- `required = {...}` — a small lookup pairing a human label ("sequence", "MIC") with the column name you set.
- `missing = [... if col not in df.columns]` — build a list of the ones that are *not* found in the real table. (This is a list comprehension: a compact for-loop that builds a list.)
- `if missing:` — an empty list is treated as "false", so this runs only when something is actually missing.
- `raise KeyError(...)` — stop the notebook and show an error. The message lists what's missing, prints the real column names, and tells you to edit the config cell.
- `has_units = UNIT_COL in df.columns` — a True/False flag: do we have a unit column at all? We use it later.
- `print("Unit column present:", has_units)` — tells you the answer.

**What you'll see:** if your names are right, just `Unit column present: True` (or `False`). If a name is wrong, a red `KeyError` telling you exactly what to fix.

In [ ]:
required = {"sequence": SEQ_COL, "MIC": MIC_COL}
missing = [f"{role} -> '{col}'" for role, col in required.items() if col not in df.columns]
if missing:
    raise KeyError(
        "Column(s) not found: " + "; ".join(missing)
        + f"\nAvailable columns: {list(df.columns)}"
        + "\nEdit SEQ_COL / MIC_COL in the Configuration cell to match."
    )
has_units = UNIT_COL in df.columns
print("Unit column present:", has_units)

### Step: keep only clean, usable rows

**What this does:** throws away rows whose sequence contains odd characters, and rows whose MIC isn't a usable number.

**Why:** real databases are messy. Some sequences contain non-standard symbols (like `X` for "unknown" or lowercase modified residues). Some MIC entries are blank or text. The biology toolbox and the models need clean, standard input, so we filter first.

**Line by line:**
- `valid_seq = re.compile(f"^[{AA}]+$")` — build a text pattern meaning "from start to end, only the 20 standard amino-acid letters, one or more of them." `^` is start, `$` is end, `[...]` is "any of these letters," `+` is "one or more."
- `def first_number(x): ...` — a small helper that pulls the first number out of a value. `re.search(r"[-+]?\d*\.?\d+", str(x))` looks for a number (optionally signed, possibly with a decimal point). If found it returns that number as a `float` (decimal); if not, it returns `np.nan`, which means "missing." This lets an entry like `">32"` still give us the number 32.
- `data = df.copy()` — work on a copy so the original `df` stays untouched.
- `data[SEQ_COL] = data[SEQ_COL].astype(str).str.strip().str.upper()` — tidy the sequence text: make it a string, remove stray spaces (`strip`), and uppercase it (`upper`), because `k` and `K` should count as the same amino acid.
- `data = data[data[SEQ_COL].apply(lambda s: bool(valid_seq.match(s)))]` — keep only rows where the sequence matches the valid pattern. `apply` runs the check on every row; `lambda s: ...` is a throwaway one-line function meaning "for a sequence s, is it valid?"
- `data["MIC_value"] = data[MIC_COL].apply(first_number)` — make a new tidy number column from the raw MIC.
- `data = data[data["MIC_value"].notna() & (data["MIC_value"] > 0)]` — keep only rows where MIC is present (`notna`) and positive. (A concentration can't be zero or negative.)
- `print(...)` — report how many peptides survived.

**What you'll see:** a single number — the count of clean peptides. Expect it to be smaller than the original row count; that drop is normal.

In [ ]:
valid_seq = re.compile(f"^[{AA}]+$")

def first_number(x):
    m = re.search(r"[-+]?\d*\.?\d+", str(x))
    return float(m.group()) if m else np.nan

data = df.copy()
data[SEQ_COL] = data[SEQ_COL].astype(str).str.strip().str.upper()
data = data[data[SEQ_COL].apply(lambda s: bool(valid_seq.match(s)))]

data["MIC_value"] = data[MIC_COL].apply(first_number)
data = data[data["MIC_value"].notna() & (data["MIC_value"] > 0)]
print("Rows with valid sequence and numeric MIC:", len(data))

### Step: put every MIC in the same unit (µg/mL)

**What this does:** converts all MIC values to one common unit so they are comparable, converting micromolar (µM) values using each peptide's molecular weight.

**Why:** MIC is reported in two different units. **µg/mL** is a mass concentration (how many micrograms of peptide per millilitre). **µM (micromolar)** is a count concentration (how many molecules). You cannot compare 16 µg/mL with 16 µM directly — that would be like comparing "16 grams of sugar" with "16 spoonfuls." To convert, you need the molecular weight (how heavy one molecule is), which we can compute from the sequence.

The conversion: **µg/mL = µM × molecular weight ÷ 1000.** (Micromolar times grams-per-mole gives the mass, and the 1000 fixes the units.)

**Line by line:**
- `def norm_unit(u): ...` — tidies a messy unit label into either `"ug/mL"`, `"uM"`, or `None` (unrecognised). It lowercases, trims spaces, and turns the Greek micro sign µ/μ into a plain `u` so `µM`, `uM`, `UM` all match.
- `data["unit_std"] = data[UNIT_COL].apply(norm_unit) if has_units else "ug/mL"` — make a tidy unit column. If the file has no unit column we assume everything is already µg/mL.
- `data["MW"] = data[SEQ_COL].apply(lambda s: peptides.Peptide(s).molecular_weight())` — compute each peptide's molecular weight from its sequence. `peptides.Peptide(s)` wraps the sequence in a smart object; `.molecular_weight()` asks it for the weight.
- `def to_ugml(row): ...` — for each row: if it is already µg/mL keep it; if it is µM convert with the formula above; otherwise return `np.nan` (can't convert).
- `data["MIC_ugml"] = data.apply(to_ugml, axis=1)` — apply that to every row. `axis=1` means "go row by row" (as opposed to column by column).
- `dropped = int(data["MIC_ugml"].isna().sum())` — count how many could not be converted.
- `data = data[data["MIC_ugml"].notna()]` — drop those unconvertible rows.
- the two `print`s — report how many were dropped and how many remain.

**What you'll see:** the number dropped (hopefully small) and the remaining count. A large drop would warn you that many rows had missing or strange units.

In [ ]:
def norm_unit(u):
    u = str(u).strip().lower().replace("\u00b5", "u").replace("\u03bc", "u")
    if u in {"ug/ml", "mg/l", "ug/cm3"}:
        return "ug/mL"
    if u in {"um", "umol/l", "umol/ml"}:
        return "uM"
    return None

data["unit_std"] = data[UNIT_COL].apply(norm_unit) if has_units else "ug/mL"
data["MW"] = data[SEQ_COL].apply(lambda s: peptides.Peptide(s).molecular_weight())

def to_ugml(row):
    if row["unit_std"] == "ug/mL":
        return row["MIC_value"]
    if row["unit_std"] == "uM":
        return row["MIC_value"] * row["MW"] / 1000.0   # uM * g/mol / 1000 = ug/mL
    return np.nan

data["MIC_ugml"] = data.apply(to_ugml, axis=1)
dropped = int(data["MIC_ugml"].isna().sum())
data = data[data["MIC_ugml"].notna()]
print(f"Dropped {dropped} row(s) with missing/unconvertible units.")
print("Remaining rows:", len(data))

## 4. What MIC means, and why 32

**MIC = Minimum Inhibitory Concentration.** It is the smallest amount of the peptide that stops visible bacterial growth in a test tube. **Lower MIC = more potent** (you need less of it to work). A peptide with MIC 4 µg/mL is stronger than one with MIC 64 µg/mL.

To do binary classification we must draw a line: above it = inactive, below it = active. We use **32 µg/mL**. Peptides needing 32 µg/mL or less are called **active (1)**; peptides needing more are **inactive (0)**.

**Important honesty point:** 32 is a *chosen* cutoff, widely used in AMP studies, but it is **not a law of nature**. A peptide at MIC 33 is barely different from one at MIC 31, yet they land on opposite sides of the line. The threshold is a practical convenience, and if you moved it the results would shift. An examiner may poke at this — it's a real limitation, not a mistake.

### Step: create the active/inactive label

**What this does:** turns the numeric MIC into the 0/1 label the models will predict, then checks how balanced the two groups are.

**Why:** the models can only learn to predict a label, so we must create one. And we must check class balance, because a lopsided split changes how we read the scores and how we train.

**Line by line:**
- `data["label"] = (data["MIC_ugml"] <= MIC_THRESHOLD).astype(int)` — the comparison gives True/False for each peptide; `.astype(int)` turns True into 1 (active) and False into 0 (inactive).
- `counts = data["label"].value_counts().sort_index()` — count how many 0s and 1s there are. `sort_index()` just puts 0 before 1 for tidy reading.
- the `print`s — show the counts.
- `ratio = counts.max() / counts.min()` — how many times bigger the larger group is. A ratio of 1 is perfectly even; 5 means one class is five times the other.
- `imbalanced = ratio >= 2` — our rule of thumb: if one class is at least twice the other, call it imbalanced. This True/False flag decides later whether to switch on `class_weight='balanced'`.
- the final `print` — tells you the ratio and the verdict.

**What you'll see:** the two counts, the ratio, and the word *balanced* or *imbalanced*. If it says imbalanced, remember that plain accuracy will be misleading (see section 2, class imbalance).

In [ ]:
data["label"] = (data["MIC_ugml"] <= MIC_THRESHOLD).astype(int)
counts = data["label"].value_counts().sort_index()
print("Class balance (0 = inactive, 1 = active):")
print(counts)
ratio = counts.max() / counts.min()
imbalanced = ratio >= 2
print(f"Imbalance ratio: {ratio:.2f} -> {'imbalanced' if imbalanced else 'balanced'}")

## 3. The biology: why these numbers should predict activity

A machine learning model only sees numbers. So we must turn each sequence into numbers that actually relate to how AMPs kill bacteria. Here is the biology behind the main ones.

Remember the killing mechanism: most AMPs work by binding to the bacterial membrane and breaking it open. Bacterial membranes (especially the outer surface of *E. coli*) are **negatively charged**. Human cell membranes are more neutral. That difference is what lets AMPs target bacteria and mostly spare our own cells.

- **Net charge** — AMPs are usually **positively charged** (cationic). Opposite charges attract, so a positive peptide is pulled toward the negative bacterial membrane. More positive charge generally means stronger attraction and often stronger activity. This is probably the single most important feature.
- **Hydrophobicity** — "water-hating." The membrane's core is oily (a lipid bilayer). A peptide needs hydrophobic (oily) parts to push into and disrupt that oily core. Too little and it can't insert; too much and it may clump or also damage human cells. So there's a sweet spot.
- **Hydrophobic moment** — this measures whether the peptide is **amphipathic**: oily on one side and charged on the other, like a coin with two faces. Imagine the peptide curled into a helix: if all the oily amino acids line up on one face and all the charged ones on the opposite face, it can lie on the membrane with its oily face dipping in and its charged face in the water. That two-faced arrangement is extremely good at tearing membranes. A high hydrophobic moment means strongly two-faced. **Crucially, this depends on the ORDER of the amino acids, not just which ones are present** — remember this for the shuffle experiment.
- **Length** — the peptide must be long enough to span or seriously disturb the membrane, but very long peptides fold up and behave differently. Most active AMPs sit in a moderate length range, so length carries real signal.
- **Isoelectric point, aliphatic index, amino-acid composition** — these are related summaries (overall charge behaviour, proportion of small oily residues, and how much of each amino acid is present). They give the model extra angles on the same underlying chemistry.

### Step: turn each sequence into features

**What this does:** for every peptide, compute the biological numbers above plus the percentage of each of the 20 amino acids, and collect them into a table.

**Why:** this is the bridge from biology to machine learning. After this step each peptide is just a row of numbers the models can learn from.

**Line by line:**
- `def features(seq): ...` — a function that takes one sequence and returns a dictionary of numbers.
- `p = peptides.Peptide(seq)` — wrap the sequence so we can ask it for properties.
- `"length": len(seq)` — number of amino acids.
- `"net_charge": p.charge()` — the overall electric charge at normal pH.
- `"hydrophobicity": p.hydrophobicity()` — average water-hating score.
- `"hydrophobic_moment": p.hydrophobic_moment() or 0.0` — the two-faced-ness score. `or 0.0` is a safety net: if the toolbox returns nothing (can happen for very short peptides) we use 0 instead of crashing.
- `"isoelectric_point": p.isoelectric_point()` — the pH at which the peptide has no net charge.
- `"aliphatic_index": p.aliphatic_index()` — proportion of small oily residues; linked to stability.
- the `for a in AA:` loop with `f[f"pct_{a}"] = 100.0 * seq.count(a) / len(seq)` — for each of the 20 amino acids, the percentage of the sequence made of that amino acid. `seq.count(a)` counts how many, divided by length, times 100.
- `X = pd.DataFrame([features(s) for s in data[SEQ_COL]], index=data.index)` — run `features` on every peptide and stack the results into a table called **X**. By convention `X` (capital) is the feature table — the model's inputs.
- `y = data["label"]` — by convention `y` (lowercase) is the labels — the answers.
- `print("Feature matrix:", X.shape)` — show the table size: (peptides, features). You should see about 26 features (6 properties + 20 amino acids).
- `X.head()` — preview the first rows.

**What you'll see:** the shape of X and a preview table of numbers. Each row is one peptide expressed entirely as numbers — the form the models need.

In [ ]:
def features(seq):
    p = peptides.Peptide(seq)
    f = {
        "length": len(seq),
        "net_charge": p.charge(),
        "hydrophobicity": p.hydrophobicity(),
        "hydrophobic_moment": p.hydrophobic_moment() or 0.0,
        "isoelectric_point": p.isoelectric_point(),
        "aliphatic_index": p.aliphatic_index(),
    }
    for a in AA:
        f[f"pct_{a}"] = 100.0 * seq.count(a) / len(seq)
    return f

X = pd.DataFrame([features(s) for s in data[SEQ_COL]], index=data.index)
y = data["label"]
print("Feature matrix:", X.shape)
X.head()

### Step: split into a training set and a test set

**What this does:** randomly sets aside 20% of the peptides as a hidden exam, and keeps 80% for the model to learn from.

**Why:** if we tested the model on the same peptides it trained on, it could just recite memorised answers (overfitting) and look great while being useless on new peptides. Holding out unseen data is the only honest way to measure learning.

**Line by line and every parameter:**
- `X_train, X_test, y_train, y_test = train_test_split(...)` — the splitter hands back four pieces: training features, test features, training labels, test labels. The order is always inputs-then-labels, train-then-test.
- `X, y` — the full features and labels we're splitting.
- `test_size=0.20` — put 20% in the test set, 80% in training. (0.20 = 20%.)
- `stratify=y` — keep the active/inactive **proportion the same** in both the training and test sets. Without this, random luck might stuff most active peptides into one side. Stratify prevents that, which matters a lot when classes are imbalanced.
- `random_state=RANDOM_STATE` — freeze the randomness so the *same* split happens every run. That's why your numbers won't change between runs.
- `print(...)` — show the sizes of each set.

**What you'll see:** two shapes, e.g. `Train: (400, 26)  Test: (100, 26)`. The test set should be about a quarter the size of the training set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 6. The three models, in plain language

We try three different kinds of model because they "think" differently, and we want to see which suits this problem. No maths — just the idea.

- **SVM (Support Vector Machine)** — imagine plotting every peptide as a dot, active dots in one cloud and inactive in another. The SVM draws the single straightest, widest "no-man's-land" boundary that best separates the two clouds, staying as far as possible from the nearest dots on each side. New peptides are classified by which side of that boundary they fall on. It's the careful line-drawer.
- **Random Forest** — a "forest" of many decision trees. One decision tree is like a game of 20 Questions: "Is charge above 3? Is it longer than 20? ..." ending in a guess. A single tree is easily fooled, so the Random Forest grows hundreds of slightly different trees and lets them **vote**. The majority answer wins. It's a committee, and committees beat lone experts.
- **Naive Bayes (Gaussian)** — it learns, for each feature, how active vs inactive peptides typically look (e.g. "active ones usually have higher charge"), then for a new peptide multiplies together the clues to judge which label is more probable. It is called "naive" because it pretends the features are independent of each other, which isn't quite true — but the shortcut often works surprisingly well and is very fast. It's the quick probability guesser.

### Step: set up the three models

**What this does:** creates the three model objects, ready but not yet trained, and decides whether to compensate for class imbalance.

**Why:** we define them once in a dictionary so the next cell can loop over all three with the same code.

**Line by line and every parameter:**
- `cw = "balanced" if imbalanced else None` — if we flagged imbalance earlier, set `cw` to the string `"balanced"`; otherwise `None` (meaning "don't adjust"). `class_weight="balanced"` tells a model to pay **more attention to the rarer class**, so it doesn't just ignore it. We pass `cw` into the models that support it.
- `models = { ... }` — a dictionary mapping each model's name to its object.
- `"SVM": make_pipeline(StandardScaler(), SVC(...))` — the SVM is wrapped in a **pipeline** with a `StandardScaler`. The scaler rescales every feature to a comparable range first. SVMs measure distances, so a big-numbers feature (like length) would otherwise drown out a small-numbers one (like charge). The pipeline does scale-then-classify as one unit. `random_state` fixes its internal randomness.
- `"Random Forest": RandomForestClassifier(class_weight=cw, random_state=...)` — the forest. It doesn't need scaling (it only asks yes/no threshold questions). `random_state` freezes which random trees get grown.
- `"Gaussian NB": GaussianNB()` — Naive Bayes. It has no `class_weight` option and needs no scaling, so it's plain.

**What you'll see:** nothing. We've only *prepared* the models; training happens next.

In [ ]:
cw = "balanced" if imbalanced else None
models = {
    "SVM": make_pipeline(StandardScaler(), SVC(class_weight=cw, random_state=RANDOM_STATE)),
    "Random Forest": RandomForestClassifier(class_weight=cw, random_state=RANDOM_STATE),
    "Gaussian NB": GaussianNB(),
}

## 5. How to read a confusion matrix and the scores

After testing, we compare each model's guesses with the true labels. A **confusion matrix** is a 2×2 box that counts the four possible outcomes. In this project:

|                   | **Predicted inactive (0)** | **Predicted active (1)** |
|-------------------|----------------------------|--------------------------|
| **Actually inactive (0)** | True Negative (TN) — correctly said inactive | False Positive (FP) — wrongly said active |
| **Actually active (1)**   | False Negative (FN) — wrongly said inactive | True Positive (TP) — correctly said active |

- **True Positive (TP):** a real antibiotic peptide, correctly flagged active. Great.
- **True Negative (TN):** a dud, correctly flagged inactive. Great.
- **False Positive (FP):** a dud the model called active. Cost: you waste a lab test on a peptide that won't work. Annoying but cheap.
- **False Negative (FN):** a real antibiotic the model called inactive. Cost: you **throw away a potential life-saving drug** and never test it. Expensive.

Now the four scores, each as a formula and in plain words:

- **Accuracy = (TP + TN) / everything.** The share of all guesses that were right. Simple, but **misleading under class imbalance** — if 90% are inactive, always guessing "inactive" scores 90% while finding zero antibiotics.
- **Precision = TP / (TP + FP).** Of the peptides the model *called active*, how many truly were. High precision = few false alarms = you don't waste lab time.
- **Recall = TP / (TP + FN).** Of the peptides that *truly are active*, how many the model caught. High recall = few missed drugs.
- **F1 = the balance of precision and recall** (their harmonic mean, `2·P·R/(P+R)`). One number that is only high when *both* precision and recall are decent. Good for imbalanced problems.

**Why recall matters most here:** a False Negative means a genuine new antibiotic is discarded forever. A False Positive just means one wasted lab test. Missing a real drug is far worse than wasting a cheap test, so we especially want **high recall** — we would rather over-flag than miss a winner.

### Step: train each model and score it

**What this does:** for each of the three models, learns from the training set, predicts the test set, prints the four scores, and draws its confusion matrix.

**Why:** this is the actual experiment — teaching each model and fairly examining it on unseen peptides.

**Line by line:**
- `results = {}` — an empty dictionary to store each model's scores for the comparison table later.
- `for name, model in models.items():` — loop over the three models one at a time.
- `model.fit(X_train, y_train)` — **fit** means *train*: the model studies the training features together with their true labels and adjusts itself to the pattern. This is the learning step.
- `pred = model.predict(X_test)` — **predict** means *guess*: feed the unseen test features in and get the model's 0/1 guesses out. Note it never saw the test labels, so this is a fair exam.
- `results[name] = { "Accuracy": accuracy_score(y_test, pred), ... }` — compare the guesses `pred` against the truth `y_test` with each scoring function and store the four numbers. `zero_division=0` just says "if a score would divide by zero, call it 0" instead of warning.
- the `print` lines — show the model's name and its four scores neatly.
- `ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["inactive", "active"])` — draw the 2×2 box of TN/FP/FN/TP from the truth and the guesses.
- `plt.title(name)` and `plt.show()` — label and display the chart.

**What you'll see:** for each model, four numbers (each between 0 and 1, higher is better) and a coloured 2×2 grid. Read the grid with section 5 in hand: you want big numbers on the diagonal (TN top-left, TP bottom-right) and small off-diagonal numbers — especially a small bottom-left box (False Negatives).

In [ ]:
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results[name] = {
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
    }
    print(f"\n=== {name} ===")
    for metric, value in results[name].items():
        print(f"{metric:10s}: {value:.3f}")
    ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=["inactive", "active"])
    plt.title(name)
    plt.show()

### Step: put all three models in one table

**What this does:** collects the four scores of all three models into a single tidy table for easy comparison.

**Why:** scrolling between three printouts is awkward; one table lets you compare at a glance.

**Line by line:**
- `scores = pd.DataFrame(results).T[[...]]` — turn the stored results into a table. `.T` flips it so each **row is a model** and each **column is a score**. The `[[...]]` fixes the column order to Accuracy, Precision, Recall, F1.
- `scores.round(3)` — show the numbers rounded to 3 decimal places so it's readable.

**What you'll see:** a 3-row table (one per model), four score columns. This is the headline result of the project — see section 7 for how to judge it.

In [ ]:
scores = pd.DataFrame(results).T[["Accuracy", "Precision", "Recall", "F1"]]
scores.round(3)

## 7. How to read that comparison table

Every score runs from 0 (hopeless) to 1 (perfect). Here's how to judge yours:

- **Pick the winner by F1**, not accuracy — F1 balances catching real drugs (recall) against not crying wolf (precision), and it isn't fooled by imbalance.
- **Rough guide:** F1 around **0.8+** is strong for this kind of task; **0.65–0.8** is decent and normal for a student project; **below ~0.6** means the features or data aren't giving the model enough to work with. Don't expect a perfect 1.0 — biology is noisy, and a perfect score usually means a bug or a data leak, not a miracle.
- **Sanity check against guessing.** If 70% of peptides are inactive, a model that always says "inactive" gets 0.70 accuracy but **0.0 recall** for active. So if a model shows high accuracy but low recall, it's secretly just guessing the majority class — a bad result dressed as a good one.
- **Look especially at recall** (section 5): for finding new antibiotics, a model with slightly lower accuracy but much higher recall is often the better choice.
- A **good** result: a clear winner with balanced, reasonably high precision and recall. A **bad** result: all scores near the majority-class proportion, or recall near zero.

### Step: which features did the Random Forest rely on?

**What this does:** asks the trained Random Forest which features it found most useful, and plots the top 10.

**Why:** this connects the result back to biology. If charge and hydrophobic moment top the list, that matches the membrane-attack theory from section 3 — a nice sanity check and a great talking point in your viva.

**Line by line:**
- `rf = models["Random Forest"]` — grab the already-trained forest from the dictionary.
- `imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)` — `feature_importances_` is a built-in list of how much each feature helped the forest's decisions (bigger = more useful). We attach the feature names (`X.columns`), sort from biggest to smallest, and keep the top 10.
- `plt.figure(figsize=(7, 4))` — start a chart 7 by 4 inches.
- `sns.barplot(x=imp.values, y=imp.index, color="steelblue")` — draw a horizontal bar chart: bar length = importance, bar label = feature name.
- `plt.xlabel(...)`, `plt.title(...)`, `plt.tight_layout()`, `plt.show()` — label, title, neaten spacing, and display.

**What you'll see:** 10 horizontal bars, longest at the top = most important feature. Check whether charge, hydrophobicity, hydrophobic moment or length appear near the top; that would support the biology.

In [ ]:
rf = models["Random Forest"]
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(10)
plt.figure(figsize=(7, 4))
sns.barplot(x=imp.values, y=imp.index, color="steelblue")
plt.xlabel("Importance")
plt.title("Top 10 features (Random Forest)")
plt.tight_layout()
plt.show()

## 8. The shuffle experiment: does ORDER matter?

Here is the clever final test. We take each peptide and **shuffle its amino acids into a random order**. Shuffling keeps the *composition* identical — the same letters, same counts, so same length, same net charge, same overall hydrophobicity, same amino-acid percentages. The one thing that changes is the **arrangement**, and therefore the **hydrophobic moment** (the two-faced-ness from section 3, which depends on order).

Then we retrain the best model on these shuffled sequences and see how much the score drops.

**What each outcome would mean:**
- **Big score drop** → arrangement matters. The model was genuinely using the *pattern* of the sequence (the amphipathic, two-faced structure), which is the real biology of membrane attack. This would be the exciting result.
- **Little or no drop** → only composition mattered. The model was really just counting amino acids, and the specific order carried little extra information *for this dataset and these features*. Still valid, but a more modest story.

Either way you learn something true about what the model latched onto. That is why this control is worth doing.

### Step: run the shuffle control

**What this does:** scrambles each sequence, rebuilds features, retrains the best-F1 model, and reports the drop in F1.

**Line by line:**
- `best_name = scores["F1"].idxmax()` — find the model with the highest F1. `idxmax` returns the row label (the model's name) of the biggest value.
- `print(...)` — announce the winner.
- `rng = np.random.default_rng(RANDOM_STATE)` — a random-number generator, seeded so the shuffling is reproducible.
- `def shuffle_seq(seq): ...` — turn the sequence into a list of letters, shuffle them in place, join back into a string.
- `shuffled = data[SEQ_COL].apply(shuffle_seq)` — shuffle every sequence.
- `X_shuf = pd.DataFrame([features(s) for s in shuffled], index=data.index)` — rebuild the feature table from the shuffled sequences, using the exact same `features` function.
- `Xs_train, Xs_test, ys_train, ys_test = train_test_split(..., stratify=y, random_state=RANDOM_STATE)` — split the shuffled data the **same way** as before (same seed, same stratification) so the comparison is fair.
- `m = clone(models[best_name])` — `clone` makes a fresh, untrained copy of the winning model with the same settings, so we train from scratch on shuffled data rather than reusing the already-trained one.
- `m.fit(Xs_train, ys_train)` — train it on shuffled features.
- `shuf_f1 = f1_score(ys_test, m.predict(Xs_test), zero_division=0)` — its F1 on the shuffled test set.
- `orig_f1 = scores.loc[best_name, "F1"]` — the winner's original F1 from before.
- the three `print`s — show original F1, shuffled F1, and the drop.

**What you'll see:** two F1 numbers and their difference. Read the difference using section 8: a big drop means order mattered; a small drop means mostly composition mattered.

In [ ]:
best_name = scores["F1"].idxmax()
print("Best model by F1:", best_name)

rng = np.random.default_rng(RANDOM_STATE)
def shuffle_seq(seq):
    chars = list(seq)
    rng.shuffle(chars)
    return "".join(chars)

shuffled = data[SEQ_COL].apply(shuffle_seq)
X_shuf = pd.DataFrame([features(s) for s in shuffled], index=data.index)

Xs_train, Xs_test, ys_train, ys_test = train_test_split(
    X_shuf, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
m = clone(models[best_name])
m.fit(Xs_train, ys_train)
shuf_f1 = f1_score(ys_test, m.predict(Xs_test), zero_division=0)
orig_f1 = scores.loc[best_name, "F1"]
print(f"Original F1 : {orig_f1:.3f}")
print(f"Shuffled F1 : {shuf_f1:.3f}")
print(f"Drop        : {orig_f1 - shuf_f1:.3f}")

## 9. Limitations — be honest about these

A good project knows its own weaknesses. State these plainly; examiners respect it.

- **Class imbalance.** If active and inactive peptides are far from 50/50, the scores can flatter the model, and it may lean toward the common class. We used `class_weight='balanced'` and watched recall and F1 to limit this, but it remains a caveat.
- **The MIC = 32 threshold is arbitrary.** It's a conventional cutoff, not a biological truth. Peptides just above and just below are nearly identical, yet get opposite labels. Different thresholds would give different results.
- **In-vitro activity ≠ a usable drug.** Killing *E. coli* in a test tube is only step one. A real drug must also be non-toxic to human cells, stable in the body, not broken down too fast, affordable to make, and so on. This model says nothing about any of that.
- **E. coli only.** The model is trained on *E. coli* data, so its predictions apply to *E. coli*. A peptide active against *E. coli* may do nothing to other bacteria, and vice versa.
- **Simple sequence features only.** We use overall properties and composition. We ignore 3-D structure, how the peptide folds, and how it behaves in a real membrane. Those matter too.
- **Dataset quality.** MIC values come from many labs with different methods, so there is measurement noise baked into the labels.

## 10. Viva preparation — likely examiner questions

**1. Why would a peptide's charge predict whether it kills bacteria?**  
Bacterial membranes are negatively charged, while AMPs are usually positively charged. Opposite charges attract, pulling the peptide onto the membrane. So higher positive charge tends to mean stronger binding and often stronger activity.

**2. What is MIC, and why did you choose 32 µg/mL as the cutoff?**  
MIC is the lowest concentration that stops bacterial growth; lower means more potent. 32 µg/mL is a common convention in AMP research for "active," but it is a chosen line, not a natural law, and moving it would change the labels.

**3. Why split the data into training and test sets?**  
To check real learning, not memorising. The model learns on the training set and is judged on unseen test peptides. Testing on data it already saw would hide overfitting and give a falsely high score.

**4. Your accuracy is high but recall is low (or vice versa) — what does that mean?**  
High accuracy with low recall usually means the model is mostly predicting the majority class and missing the active peptides. Recall is the share of truly active peptides it caught, so low recall means missed potential drugs — the costly error here.

**5. Why do you report F1 instead of just accuracy?**  
Accuracy is misleading when classes are imbalanced — always guessing the common class can score high while finding nothing useful. F1 combines precision and recall, so it is only high when the model is genuinely good at both.

**6. What does the shuffle experiment actually test?**  
It keeps each peptide's amino-acid composition but randomises the order, so only the arrangement (and the hydrophobic moment) changes. A big score drop means the model used sequence pattern (real structure); a small drop means it mostly used composition.

**7. Why did you try three different models?**  
They make decisions in different ways — SVM draws a boundary, Random Forest votes across many decision trees, Naive Bayes multiplies probabilities. Comparing them shows which style fits this data best and guards against trusting one model blindly.

**8. What are the main limitations of this project?**  
The MIC threshold is arbitrary, the classes may be imbalanced, and the model is trained on *E. coli* only. Most importantly, killing bacteria in a test tube doesn't make something a drug — toxicity, stability and delivery all still matter.